# Session Processing

Wavesurfer → session data (licks, rewards, treadmill, bouts).

Outputs go to `stroke_orofacial_pipeline_outputs/spout_behavior/` on the MICROSCOPE server.

**Phase 6.7 update:** orchestration now flows through `SessionManifest`. Cell 3 builds one manifest per (animal, date); cell 4 passes each manifest to `process_session()`.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

# Dates to process per animal. Set to None to process all dates in the
# animal's session range (from configs/animals.yaml).
# Or provide an explicit list of datetime.date objects.
DATES = None  # None = all dates in session range

# Overwrite mode: "skip" (default), "prompt", or "force"
OVERWRITE = "skip"

In [ ]:
import datetime
import logging

from stroke_orofacial.animals import load_animals
from stroke_orofacial.config_loader import load_defaults, load_session_overrides
from stroke_orofacial.manifest import build_manifest
from stroke_orofacial.paths import PathResolver
from stroke_orofacial.spout_behavior.session_processing import process_session

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

resolver = PathResolver()
animals_config = load_animals()
defaults = load_defaults()
overrides_config = load_session_overrides()

In [ ]:
# ============================================================
# Build a SessionManifest per (animal, date).
# Surfaces wavesurfer / video / DLC discovery in the notebook state
# before any processing runs, so it's easy to inspect what will be
# processed (and what was filtered out).
# ============================================================

manifests = []  # list of (animal_id, date, manifest-or-None) tuples

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    for date in dates:
        try:
            manifest = build_manifest(animal_id, date, resolver, info)
        except FileNotFoundError as e:
            # Most common cause: DLC inference hasn't run for this date yet,
            # which build_manifest treats as a hard error. Phase 2 (spout
            # behavior) only needs wavesurfer; if you want to process such
            # sessions, build a manifest manually with empty dlc_h5_files
            # and call process_session(manifest, resolver, ...) directly.
            print(f"  build_manifest failed for {animal_id}/{date}: {e}")
            manifests.append((animal_id, date, None))
            continue
        manifests.append((animal_id, date, manifest))

n_built = sum(1 for _, _, m in manifests if m is not None)
n_failed = sum(1 for _, _, m in manifests if m is None)
print(f"Built {n_built} manifests, {n_failed} failed.")

In [ ]:
# ============================================================
# Process each session by passing its manifest to process_session().
# ============================================================

results = []
for animal_id, date, manifest in manifests:
    if manifest is None:
        results.append({})
        continue
    try:
        result = process_session(
            manifest, resolver,
            overwrite=OVERWRITE,
            defaults=defaults,
            overrides_config=overrides_config,
        )
        results.append(result)
    except NotImplementedError as e:
        print(f"  Skipping {animal_id}/{date}: {e}")
        results.append({})
    except FileNotFoundError as e:
        print(f"  Skipping {animal_id}/{date}: {e}")
        results.append({})

n_processed = sum(1 for r in results if r)
n_skipped = sum(1 for r in results if not r)
print(f"\nDone: {n_processed} processed, {n_skipped} skipped/missing.")